## 字符串输出解析器StrOutputParser

In [15]:
from langchain_core.output_parsers import StrOutputParser
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

# 使用向量模型
llm = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

message=llm.invoke('什么是大语言模型')
# 方式一：自己调用输出结果的content
# print(message.content)


# 方式二：使用StrOutputParser
parser = StrOutputParser()
result=parser.invoke( message)
print(result)





大语言模型（Large Language Model，简称LLM）是一种基于深度学习的人工智能模型，它通过在大规模文本数据上进行训练，学习语言的结构、语法、语义和上下文关系，从而能够生成或理解自然语言。大语言模型通常具有数十亿甚至更多的参数，这使得它们能够捕捉到语言的复杂模式，并在多种自然语言处理任务上表现出色，如文本生成、机器翻译、问答、对话系统、文本摘要等。

大语言模型的训练通常采用自监督学习的方式，即在没有人工标注的数据上进行训练，通过预测文本中的下一个词或掩码词来学习语言表示。这种训练方式使得大语言模型能够从海量的互联网文本中学习到丰富的语言知识，而不需要针对每个具体任务进行专门的标注数据。

近年来，随着计算资源的提升和深度学习技术的发展，大语言模型的规模和性能不断提升，成为自然语言处理领域的重要研究方向。


## JSON解析器JsonOutputParser

In [3]:
# 方式一：在提示词中说明输出的格式，解析成功与否完全依赖模型是否输出合法 JSON。
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

# 使用模型
llm = ChatOpenAI(
    model='tongyi-xiaomi-analysis-flash',
    temperature=0.6
)
chat_messages=ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}"),
    ("human", "{question}"),
])


json_parser=JsonOutputParser()

chain=chat_messages|llm|json_parser
result=chain.invoke({"role": "助手", "question": "AI会替代人类？需要返回一个JSON格式的数据"})
print( result)



{'topic': 'AI是否会替代人类', 'analysis': {'ai_abilities': ['数据处理和分析', '自动化任务执行', '语言理解和生成', '图像和语音识别'], 'areas_of_competence': ['重复性和规则导向的任务', '数据分析和预测', '内容创作和生成', '客户服务和交互'], 'areas_of_human_competence': ['创造力和抽象思维', '情感理解和表达', '复杂决策和判断', '道德和伦理判断'], 'analysis_summary': 'AI在某些领域已经展现出强大的能力，特别是在数据处理、自动化任务执行和内容生成方面。然而，AI在情感理解、复杂决策和道德判断等方面仍无法完全替代人类。AI更倾向于在辅助和增强人类能力方面发挥重要作用，而不是完全替代人类。'}}


In [1]:
# 方式二：自动生成格式说明，输出解析后自动是
# 缺点：需要额外定义一个类。 模型通用
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import os
import dotenv
dotenv.load_dotenv()


# 1. 定义数据结构 (使用 Pydantic) 模型输出的键值
class Joke(BaseModel):
    setup: str = Field(description="笑话的设置部分")
    punchline: str = Field(description="笑话的笑点部分")

# 2. 初始化解析器，并将结构传入
parser = JsonOutputParser(pydantic_object=Joke)

# 3. 将解析器的指令添加到模板中
prompt = ChatPromptTemplate.from_template(
    "回答问题。\n{format_instructions}\n问题: {query}"
)
# format_instructions对应的是from_template里面的format_instructions
# get_format_instructions()：这是该解析器的关键。它会自动生成一段系统提示词（Prompt），告诉 LLM 以特定的 JSON 格式返回结果。
prompt = prompt.partial(format_instructions=parser.get_format_instructions())
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model =  ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)
# 4. 构建链
chain = prompt | model | parser

# 5. 执行
result = chain.invoke({"query": "讲一个关于狗的笑话。"})

print(result)

{'setup': '为什么狗不能玩捉迷藏？', 'punchline': '因为它们总是能找到‘骨’（窟）藏的地方。'}


In [5]:
# 方式三：在二的基础上简洁写法
# 模型可能不兼容这张写法 with_structured_output
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 1. 定义数据结构（和之前一样）
class Joke(BaseModel):
    setup: str = Field(description="笑话的设置部分")
    punchline: str = Field(description="笑话的笑点部分")

# 2. 初始化模型
model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

# 3. 绑定结构化输出
structured_model = model.with_structured_output(Joke)

# 4. 直接调用，结果就是解析好的字典或 Pydantic 对象
result = structured_model.invoke("讲一个关于猫的笑话。")
print(result)      # 可以直接访问属性



ValidationError: 1 validation error for Joke
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='好的，这里有一个...“鼠标”点击！', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid

## XMLOutputParser XML输出解析器的使用

In [4]:
# 方式一：在提示词中说明输出的格式，解析成功与否完全依赖模型是否输出合法 XML。
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

# 使用模型
llm = ChatOpenAI(
    model='tongyi-xiaomi-analysis-flash',
    temperature=0.6
)
chat_messages=ChatPromptTemplate.from_template(
    "你是一个{role}，\n问题: {question}"
)

xml_parser=XMLOutputParser()
chain=chat_messages|llm|xml_parser
# 这是已经把模型输出的XML转成字典了
result=chain.invoke({"role": "助手", "question": "AI会替代人类？返回一个XML格式的数据"})
print( result)


{'Analysis': [{'Question': 'AI会替代人类？'}, {'AnalysisContent': [{'Introduction': '随着人工智能技术的快速发展，关于AI是否会替代人类的讨论日益激烈。'}, {'Discussion': [{'Point': 'AI在特定任务上展现出了超越人类的能力，如数据分析、图像识别和自动化控制等。'}, {'Point': '在一些重复性和标准化程度高的工作中，AI可以更高效地完成任务，如数据录入、客户服务机器人等。'}, {'CounterPoint': '然而，AI在创造性和需要情感理解的领域，如艺术创作、心理咨询等，仍然需要人类的参与。'}, {'CounterPoint': 'AI的发展更多的是与人类协作，增强人类的能力，而不是完全替代人类。'}]}]}, {'Conclusion': 'AI不会完全替代人类，而是与人类形成一种协作关系，共同推动社会的发展。'}]}


In [3]:
# 方式二：使用XMLOutputParser的get_format_instructions方法自动转成XML
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI
parser=XMLOutputParser()
prompt=ChatPromptTemplate.from_template(
    '''
        前面的回答问题，后面的对应问题
    '''
     "回答以下问题：\n{format_instructions}\n问题: {query}"
)
prompt=prompt.partial(format_instructions=parser.get_format_instructions())
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

# 使用模型
model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-flash',
    temperature=0.6
)

# 这样输出的是原始的XML数据
result=model.invoke(prompt.invoke({"query": "告诉我一个英雄的名字和他的事迹"}))

print(result.content)


<response>
  <hero>
    <name>孙悟空</name>
    <achievement>
      <legend>大闹天宫，保护唐僧取经，最终被封为斗战胜佛</legend>
      <detail>孙悟空是中国古典小说《西游记》中的主要角色，他拥有七十二变和筋斗云的神通，曾大闹天宫，后被如来佛祖镇压在五行山下。在唐僧师徒四人的取经路上，他保护唐僧战胜各种妖魔鬼怪，最终成功取得真经，被封为斗战胜佛。</detail>
    </achievement>
  </hero>
</response>


In [2]:
# 举例三：正式版本，可以提取XML数据，XMLOutputParser不会直接返回XML数据，会解析XML并转成字典数据
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import XMLOutputParser
import dotenv
import os
# 1. 初始化解析器
# 可以指定 tags 参数来明确你要提取哪些标签
# parser = XMLOutputParser(tags=["name", "info"])
parser = XMLOutputParser()

# 2. 获取格式说明并构建提示词
prompt = ChatPromptTemplate.from_template(
    "回答关于角色的问题。\n{format_instructions}\n问题: {query}"
)
# 这里的format_instructions对应的是from_template里面的format_instructions
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 3. 构建链
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-flash',
    temperature=0.6
)
chain = prompt | model | parser

# 4. 执行链
result = chain.invoke({"query": "告诉我一个英雄的名字和他的事迹"})

print(result)

{'response': [{'hero': [{'name': '孙悟空'}, {'achievements': [{'achievement': '大闹天宫，挑战天庭权威'}, {'achievement': '三打白骨精，保护唐僧取经'}, {'achievement': '收服猪八戒、沙僧等徒弟，共同西天取经'}]}]}]}
